In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 12})
import os
import time
import sys
sys.path.append('../../../corriscope')
from corriscope.fpga_array import FPGAArray

In [ ]:
ca = FPGAArray('crs 50:1', stderr_log_level='warn', mode='corr8')

In [ ]:
i = ca.ib[0]
ucap = i.UCAP
ucap.MODE = 0
u_receiver = i.get_data_receiver(port_number=0)

In [ ]:
NPOLS = 8
M = 2**14
NBINS = M//2
fs = 3200 # MHz
f_res = fs/M # MHz
f = f_res*np.arange(NBINS)
n_cap_fft = 1000

# Calibrate Digital Gains

In [ ]:
# Set the channelizer to capture 32+32-bit FFT data
i.set_channelizer(adc_mode=None,
                     adc_sampling_mode=None,
                     adc_bandwidth=None,
                     adcdaq_mode=None,
                     data_source='adc',
                     function=None,
                     funcgen_left_shift=0,
                     funcgen_right_shift=0,
                     # freq_test_bins=None,  # will be passed into function_kwargs
                     fft_bypass=False,
                     fft_shift=0b00000000000000,
                     scaler_bypass=True,
                     scaler_cap_data_type=4,
                     scaler_bypass_data_type=None,
                     postscaler=None,
                     scaler_eight_bit=None,
                     scaler_rounding_mode=None,
                     symmetric_saturation=None,
                     zero_on_sat=None,
                     prober_user_flags=None,
                     offset_binary_encoding=None,
                     local_sync=True,
                     channels=list(np.arange(NPOLS)),
                     gains = None # 1*2**32
                     )

# Start data capture for even bins
i.start_data_capture(period = 0.02, 
                     source = 'scaler',
                     mode = ucap.MODE,
                     channels = list(np.arange(NPOLS)),
                     data_type = 'fft_2x',
                     select = True
                     )

# Capture data!
t1 = time.time()
t, d_fft_32_32_raw, c = u_receiver.read_raw_frames(format = '32+32', 
                                                   ncap = n_cap_fft, # n_cap_fft//n_frames_per_capture(), 
                                                   split = True,
                                                   verbose = 0
                                                   )
t2 = time.time()
print(f'Took {t2 - t1} s to capture {n_cap_fft} frames')

# **WARNING!**

# The following line assumes that the `period` as set above ensures 
# that the first frame of any capture is always even, ensuring that 
# even bins are always captured first.

# Stitch together even and odd bins
s = d_fft_32_32_raw.shape
d_fft = d_fft_32_32_raw.reshape(s[0], -1, 2, s[2]).reshape(s[0], -1, s[2]*2, order = 'F')
d_fft_mean_mag = np.mean(np.abs(d_fft), axis = 1)
d_fft_mean_psd = np.mean(np.abs(d_fft)**2, axis = 1)

In [ ]:
def make_bin_map():
    f_nat = np.arange(8192)
    f_rev = (f_nat>>11) | ((f_nat & 0b11111111111) << 2)
    frev2 = (f_rev & 0b1111111111100) | ((f_rev + np.arange(8192)) & 0b11)
    return frev2

def make_inv_bin_map():
    return np.argsort(make_bin_map())

bin_map = make_bin_map()
inv_bin_map = make_inv_bin_map()

In [ ]:
plt.figure(figsize = (12, 10))
plt.suptitle('raw FFT, 32+32 bits')

for n in range(NPOLS):

    plt.subplot(8, 3, 3*n + 1)
    if n == 0:
        plt.title('real')
    plt.plot(f, d_fft[n, 0, bin_map].real)
    plt.xticks(np.arange(0, 1600 + 400, 400))
    plt.xlim(0, 1600)
    plt.ylabel(f'ADC{n + 1}')
    plt.grid()

    if n + 1 == 8:
        plt.xlabel('frequency [MHz]')
    
    plt.subplot(8, 3, 3*n + 2)
    if n == 0:
        plt.title('imaginary')
    plt.plot(f, d_fft[n, 0, bin_map].imag, color = 'C1')
    plt.xticks(np.arange(0, 1600 + 400, 400))
    plt.xlim(0, 1600)
    plt.grid()

    if n + 1 == 8:
        plt.xlabel('frequency [MHz]')
    
    plt.subplot(8, 3, 3*n + 3)
    if n == 0:
        plt.title('power')
    plt.semilogy(f, np.mean(np.abs(d_fft[n, :, bin_map])**2, axis = 1), color = 'C2')
    plt.xticks(np.arange(0, 1600 + 400, 400))
    plt.xlim(0, 1600)
    plt.grid()

    if n + 1 == 8:
        plt.xlabel('frequency [MHz]')

plt.tight_layout()

In [ ]:
target = 1.5*np.sqrt(2)
target_full = target*2**44

In [ ]:
gain = target_full/d_fft_mean_mag

In [ ]:
# Treat infinities, making them very small but finite
gain[np.where(gain == np.inf)] = 1e-10

In [ ]:
# Set the digital gains
for n in range(NPOLS):
    i.chan[n].SCALER.set_gain_table(gain[n], bank = 0, gain_timestamp = None, log_gain = None)

In [ ]:
# Read back the quantized digital gains

read_gains_corrected = np.zeros((NPOLS, NBINS))

for n in range(NPOLS):
    read_gains = np.array(i.chan[n].SCALER.get_gain_table(bank = 0)) & 0xFFFF
    read_gains_common = 2**i.chan[n].SCALER.SHIFT_LEFT
    read_gains_corrected[n] = read_gains_common*((read_gains & 0x7FF) << (read_gains >> 11)).flatten()

# Treat any zeros
read_gains_corrected[np.where(read_gains_corrected) == 0.] = np.nan

In [ ]:
plt.figure(figsize = (12, 4))
plt.semilogy(f, gain[0, bin_map])
plt.semilogy(f, read_gains_corrected[0, bin_map])
plt.grid()
plt.tight_layout()

# Read Correlated Data

In [ ]:
# Get correlator receiver
cr = i.get_corr_receiver()

In [ ]:
#Setup correlator and channelizer
i.set_channelizer(data_source = 'adc', fft_bypass = False, scaler_bypass = False, offset_binary_encoding = False, gains = None) # 1<<31)

n_firmware_frames = 16384
NCORR_PROD = 36
i.start_correlator(integration_period = n_firmware_frames)

In [ ]:
n_software_frames = 119 # roughly 10 s integration with 16384 firmware frames

_, d_corr, counts, sat = cr.read_corr_frames(soft_integ_period = n_software_frames, 
                                          number_of_results = 1, 
                                          flush = True,
                                          verbose = 1
                                          )

In [ ]:
d_corr.shape

In [ ]:
p = 0

plt.figure(figsize = (12, 6))
plt.title(f'raw correlator output, product {p}, {n_software_frames} software frames')
plt.semilogy(f, np.abs(d_corr[0, :, p]))
plt.ylabel('power [arb.]')
plt.xlabel('frequency [MHz]')
plt.grid()
plt.tight_layout()

In [ ]:
N = 8
prod = [ (i, (j + i) % N) if i < N - j else ((j + i) % N, i - 1) for i in range(N + 1) for j in range(N//2)] 

In [ ]:
read_gains_corrected.shape

In [ ]:
counts_good = counts[0] == n_software_frames*len(f)
d_corr_fft_units = np.zeros(d_corr.shape, dtype = np.complex128)

if counts_good:
    
    corr_to_sclr_units = 1/(n_software_frames*n_firmware_frames)
    d_corr_sclr_units = d_corr*corr_to_sclr_units
    
    for n in range(NCORR_PROD):
        p0 = prod[n][0]
        p1 = prod[n][1]
    
        try:
            corr_sclr_units_to_corr_fft_units = (2**44)**2/(read_gains_corrected[p0, bin_map]*read_gains_corrected[p1, bin_map])
            d_corr_fft_units[..., n] = d_corr_sclr_units[..., n]*corr_sclr_units_to_corr_fft_units[None, :]
        except:
            continue 
else:
    print(counts)
    raise ValueError('Check your counts array. You may have dropped packets.')

In [ ]:
plt.figure(figsize = (12, 8))

plt.subplot(211)
plt.title('comparison of FFT and autocorrelation spectra')
plt.semilogy(f[1:], d_fft_mean_psd[0, bin_map][1:], label = 'FFT')
plt.semilogy(f[1:], d_corr_fft_units[0, :, 0][1:], label = 'auto')
plt.ylabel('power [arb.]')
plt.xticks(np.arange(0, 1600 + 100, 100))
plt.grid()
plt.legend(loc = 'lower right')

plt.subplot(212)
plt.plot(f[1:], 100*((d_corr_fft_units[0, :, 0] - d_fft_mean_psd[0, bin_map])/d_fft_mean_psd[0, bin_map])[1:], label = 'corr', color = 'C2')
plt.ylabel('fractional deviation \nfrom raw FFT [%]')
plt.xlabel('frequency [MHz]')
plt.xticks(np.arange(0, 1600 + 100, 100))
plt.grid()

plt.tight_layout()

In [ ]:
plt.figure(figsize = (12, 6))

# plt.title('comparison of FFT, scaler, and autocorrelation spectra')
# plt.semilogy(f[1:], d_fft_mean_psd[0, bin_map][1:], label = 'FFT')
plt.plot(f[1:], np.angle(d_corr_fft_units[0, :, 1][1:]), label = 'cross')
plt.plot(f[1:], np.angle(d_corr_fft_units[0, :, 0][1:]), label = 'auto')
plt.xticks(np.arange(0, 1600 + 100, 100))
# plt.xlim(0, 100)
plt.ylabel('phase [rad.]')
plt.xlabel('frequency [MHz]')
plt.grid()
plt.legend(loc = 'lower right')
plt.tight_layout()